In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import fbeta_score,classification_report, average_precision_score

In [2]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

In [3]:
# Prepare for chunked reading
cols = features + [target]
chunk_size = 1_000_000
sample_size = 1_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=cols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=cols, fill_value=0) # Ensure all columns are present

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk # Yield the chunk for processing
        
# Count classes
not_cancelled_total = 0
cancelled_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    not_cancelled_total += (y_chunk == 0).sum()
    cancelled_total += (y_chunk == 1).sum()

total_rows = not_cancelled_total + cancelled_total

print(f"Train not cancelled: {not_cancelled_total:,}")
print(f"Train cancelled: {cancelled_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide sample sizes
not_cancelled_sample_size = int(sample_size * not_cancelled_total / total_rows)
cancelled_sample_size = sample_size - not_cancelled_sample_size

print(f"Sampling not cancelled: {not_cancelled_sample_size:,}")
print(f"Sampling cancelled: {cancelled_sample_size:,}")

Train not cancelled: 43,903,090
Train cancelled: 4,479,366
Train total:     48,382,456
Sampling not cancelled: 907,417
Sampling cancelled: 92,583


In [4]:
# Generate random numbers for sampling (default_rng is faster)
rng = np.random.default_rng(random_state)

X_not_cancelled_parts = []
y_not_cancelled_parts = []
X_cancelled_parts = []
y_cancelled_parts = []

not_cancelled_seen = 0
cancelled_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    # Create masks for cancelled and not cancelled
    not_cancelled_mask = y_chunk == 0
    cancelled_mask = y_chunk == 1
    
    # Split the chunk into cancelled and not cancelled
    X_not_cancelled = X_chunk.loc[not_cancelled_mask]
    y_not_cancelled = y_chunk.loc[not_cancelled_mask]

    X_cancelled = X_chunk.loc[cancelled_mask]
    y_cancelled = y_chunk.loc[cancelled_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = not_cancelled_sample_size - sum(len(p) for p in y_not_cancelled_parts)
    pos_remaining_needed = cancelled_sample_size - sum(len(p) for p in y_cancelled_parts)

    neg_remaining_total = not_cancelled_total - not_cancelled_seen
    pos_remaining_total = cancelled_total - cancelled_seen

    # Sample not cancelled
    if neg_remaining_needed > 0 and len(X_not_cancelled) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_not_cancelled)) < p_neg

        X_not_cancelled_parts.append(X_not_cancelled.loc[keep_neg])
        y_not_cancelled_parts.append(y_not_cancelled.loc[keep_neg])
        
    # Sample cancelled
    if pos_remaining_needed > 0 and len(X_cancelled) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_cancelled)) < p_pos

        X_cancelled_parts.append(X_cancelled.loc[keep_pos])
        y_cancelled_parts.append(y_cancelled.loc[keep_pos])

    not_cancelled_seen += len(X_not_cancelled)
    cancelled_seen += len(X_cancelled)

# Combine sampled parts
X_train_sample = pd.concat(X_not_cancelled_parts + X_cancelled_parts, axis=0)
y_train_sample = pd.concat(y_not_cancelled_parts + y_cancelled_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(copy=False)

In [5]:
# Baseline: Logistic Regression from existing prepared data (no re-reading CSV).
X_values = np.asarray(X_train_sample, dtype=np.float32)
y_values = np.asarray(y_train_sample)

# Per-feature means for NaN imputation
column_means = np.nanmean(X_values, axis=0).astype(np.float32)
column_means = np.where(np.isnan(column_means), 0.0, column_means).astype(np.float32)

model = SGDClassifier(loss="log_loss", random_state=42)

# Train in chunks to avoid memory issues.
for i, start in enumerate(range(0, len(X_values), 1_000_000), start=1):
    end = start + 1_000_000
    X_chunk_np = X_values[start:end]
    y_chunk_np = y_values[start:end]

    if np.isnan(X_chunk_np).any():
        X_chunk_np = np.where(np.isnan(X_chunk_np), column_means, X_chunk_np).astype(np.float32, copy=False)

    if i == 1:
        model.partial_fit(X_chunk_np, y_chunk_np, classes=np.array([0, 1], dtype=np.int8))
    else:
        model.partial_fit(X_chunk_np, y_chunk_np)

    print(f"Train chunk {i}: {X_chunk_np.shape}")

Train chunk 1: (999797, 35)


In [6]:
# Evaluate CSV split in chunks
def evaluate_split_logreg(name, file_path):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    # Read and evaluate in chunks to avoid memory issues
    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        # Impute NaNs in the chunk using precomputed column means
        if np.isnan(X_chunk_np).any():
            X_chunk_np = np.where(np.isnan(X_chunk_np), column_means, X_chunk_np).astype(np.float32, copy=False)

        y_true_np = y_chunk.to_numpy(copy=False)
        y_pred_np = model.predict(X_chunk_np)

        # Get probabilities for PR-AUC calculation
        if hasattr(model, "predict_proba"):
            y_prob_np = model.predict_proba(X_chunk_np)[:, 1]
        else:
            scores = model.decision_function(X_chunk_np)
            y_prob_np = 1.0 / (1.0 + np.exp(-scores))

        y_true_parts.append(y_true_np)
        y_pred_parts.append(y_pred_np)
        y_prob_parts.append(y_prob_np)

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)

    print(f"\n{name}")
    print("Rows:", y_true_all.shape[0])
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_true_all, y_prob_all), 4))


evaluate_split_logreg("Validation", "val_data.csv")
evaluate_split_logreg("Test", "test_data.csv")


Validation
Rows: 10367669
F2 Score: 0.0
              precision    recall  f1-score   support

           0      0.900     1.000     0.948   9333880
           1      0.000     0.000     0.000   1033789

    accuracy                          0.900  10367669
   macro avg      0.450     0.500     0.474  10367669
weighted avg      0.811     0.900     0.853  10367669

PR-AUC: 0.0997

Test
Rows: 10367670
F2 Score: 0.0
              precision    recall  f1-score   support

           0      0.878     1.000     0.935   9105889
           1      0.000     0.000     0.000   1261781

    accuracy                          0.878  10367670
   macro avg      0.439     0.500     0.468  10367670
weighted avg      0.771     0.878     0.821  10367670

PR-AUC: 0.1217
